# Pegangan UTS Data Mining II — Image

**Cara pakai: import → konfigurasi analisis → sisakan SATU chunk loading → preprocessing → analisis.**

Hapus semua chunk loading yang tidak dipakai, termasuk judul dan seluruh cell kodenya.
Setelah itu baru gunakan Run All. Tidak ada variabel `SOURCE` untuk memilih input.
Semua alternatif menghasilkan `df_inputs` dan `load_failed`; bagian berikutnya tetap sama.

| Chunk | Input |
|---|---|
| A | Folder gambar lokal, termasuk subfolder |
| B | ZIP berisi gambar |
| C | CSV/Excel berisi lokasi gambar |
| D | CSV/Excel berisi kolom piksel |
| E | Dataset gambar dari Kaggle |
| F | MNIST dari TensorFlow |
| G | Dataset Kaggle berbentuk CSV/Excel piksel |

Lokasi `/content/...` adalah contoh Colab. Ganti sesuai lokasi data di komputer.
Notebook belum dijalankan pada dataset. Jika soal menentukan k, isi `K_FIXED`.


## 1. Import umum
Install package sekali jika diperlukan. Kagglehub dan TensorFlow dicantumkan hanya di chunk masing-masing.


In [ ]:
# %pip install numpy pandas matplotlib scikit-learn pillow openpyxl


In [ ]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score


## 2. Pengaturan analisis bersama
`IMAGE_SIZE=(lebar, tinggi)`. Gunakan `L` untuk grayscale atau `RGB` untuk warna. Pengaturan path berada di setiap chunk loading.


In [ ]:
IMAGE_SIZE = (28, 28)
COLOR_MODE = "L"
MAX_ITEMS = None  # misalnya 2000 untuk latihan cepat
RANDOM_STATE = 42

K_FIXED = None
K_CANDIDATES = [2, 3, 4, 5, 6, 7, 8, 10]
SILHOUETTE_SAMPLE = 1000  # None untuk seluruh data
USE_MINIBATCH = False


# PILIH SATU CHUNK LOADING — hapus alternatif lainnya

Untuk label dari folder, aktifkan `LABEL_FROM_FOLDER` dan arahkan folder data langsung ke
kategori, misalnya `data/train/` yang berisi `kucing/` dan `anjing/`. Label diambil dari
subfolder pertama di bawah folder data. Biarkan nonaktif jika nama folder bukan kategori.


## CHUNK A — Folder gambar lokal

Ganti `DATA_DIR`. Pencarian mencakup seluruh subfolder. Setelah chunk ini, lanjut ke PREPROCESSING BERSAMA.


In [ ]:
DATA_DIR = Path("/content/data/image")
LABEL_FROM_FOLDER = False


In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Folder tidak ditemukan: {DATA_DIR}")
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
paths = sorted(p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
labels = []
for file in paths:
    parts = file.relative_to(DATA_DIR).parts
    labels.append(parts[0] if LABEL_FROM_FOLDER and len(parts) > 1 else None)
df_inputs = pd.DataFrame({"image": paths, "path": [str(p) for p in paths], "label": labels})
load_failed = []
print("Gambar ditemukan:", len(df_inputs))
display(df_inputs[["path", "label"]].head())


## CHUNK B — ZIP gambar

Ganti ZIP dan folder ekstraksi. `IMAGE_SUBFOLDER=""` mencari dari akar ekstraksi; isi jika ZIP memiliki folder pembungkus. Gunakan folder ekstraksi berbeda untuk ZIP berbeda.


In [ ]:
ZIP_PATH = Path("/content/image.zip")
EXTRACT_DIR = Path("/content/image_extracted")
IMAGE_SUBFOLDER = ""  # contoh: "dataset/train"
LABEL_FROM_FOLDER = False

with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(EXTRACT_DIR)
print("Isi hasil ekstraksi:", [p.name for p in EXTRACT_DIR.iterdir()][:20])
DATA_DIR = EXTRACT_DIR / IMAGE_SUBFOLDER


In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Folder tidak ditemukan: {DATA_DIR}")
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
paths = sorted(p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
labels = []
for file in paths:
    parts = file.relative_to(DATA_DIR).parts
    labels.append(parts[0] if LABEL_FROM_FOLDER and len(parts) > 1 else None)
df_inputs = pd.DataFrame({"image": paths, "path": [str(p) for p in paths], "label": labels})
load_failed = []
print("Gambar ditemukan:", len(df_inputs))
display(df_inputs[["path", "label"]].head())


## CHUNK C — CSV/Excel lokasi gambar

Tabel berisi lokasi file, bukan piksel. Path relatif dihitung dari folder tabel, atau dari `FILE_BASE_DIR` jika diisi. File gambar harus tersedia di komputer.


In [ ]:
TABLE_PATH = Path("/content/image_paths.csv")
PATH_COLUMN = "filename"
LABEL_COLUMN = None  # contoh: "label"
FILE_BASE_DIR = None  # contoh: Path("/content/data/image")
CSV_SEP = ","
CSV_ENCODING = "utf-8-sig"

if TABLE_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df_table = pd.read_excel(TABLE_PATH)  # .xls lama membutuhkan xlrd
else:
    df_table = pd.read_csv(TABLE_PATH, sep=CSV_SEP, encoding=CSV_ENCODING)
display(df_table.head())
if PATH_COLUMN not in df_table.columns:
    raise ValueError(f"Periksa PATH_COLUMN. Kolom: {list(df_table.columns)}")
if LABEL_COLUMN is not None and LABEL_COLUMN not in df_table.columns:
    raise ValueError("LABEL_COLUMN tidak ditemukan.")


In [ ]:
missing = df_table[PATH_COLUMN].isna() | df_table[PATH_COLUMN].astype(str).str.strip().eq("")
load_failed = [{"path": f"row_{i}", "error": "Lokasi gambar kosong"} for i in df_table.index[missing]]
df_table = df_table.loc[~missing].copy()
base_dir = TABLE_PATH.parent if FILE_BASE_DIR is None else Path(FILE_BASE_DIR)
paths = [Path(str(value).strip()) for value in df_table[PATH_COLUMN]]
paths = [p if p.is_absolute() else base_dir / p for p in paths]
labels = df_table[LABEL_COLUMN].to_numpy() if LABEL_COLUMN is not None else [None] * len(paths)
df_inputs = pd.DataFrame({"image": paths, "path": [str(p) for p in paths], "label": labels})
print("Baris tanpa lokasi dibuang:", missing.sum())
display(df_inputs[["path", "label"]].head())


## CHUNK D — CSV/Excel piksel

Isi bentuk gambar dan keluarkan ID/label dari kolom piksel. Urutan kolom harus sesuai urutan piksel. `PIXEL_MAX=1.0` jika data sudah 0–1; 255.0 untuk piksel 0–255.


In [ ]:
TABLE_PATH = Path("/content/image_pixels.csv")
LABEL_COLUMN = None
CSV_SEP = ","
CSV_ENCODING = "utf-8-sig"
PIXEL_SHAPE = (28, 28)  # (tinggi, lebar) atau (tinggi, lebar, 3)
PIXEL_COLUMNS = None  # daftar nama kolom piksel; None memilih selain kolom yang dikecualikan
NON_PIXEL_COLUMNS = ["id", "Id"]
PIXEL_MAX = 255.0

if TABLE_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df_table = pd.read_excel(TABLE_PATH)
else:
    df_table = pd.read_csv(TABLE_PATH, sep=CSV_SEP, encoding=CSV_ENCODING)
display(df_table.head())
if LABEL_COLUMN is not None and LABEL_COLUMN not in df_table.columns:
    raise ValueError("LABEL_COLUMN tidak ditemukan.")
if MAX_ITEMS is not None and len(df_table) > MAX_ITEMS:
    print("Sampling tabel piksel:", MAX_ITEMS, "dari", len(df_table))
    df_table = df_table.sample(n=MAX_ITEMS, random_state=RANDOM_STATE)


In [ ]:
excluded = NON_PIXEL_COLUMNS + ([LABEL_COLUMN] if LABEL_COLUMN is not None else [])
pixel_cols = PIXEL_COLUMNS if PIXEL_COLUMNS is not None else [c for c in df_table.columns if c not in excluded]
if len(PIXEL_SHAPE) not in {2, 3} or (len(PIXEL_SHAPE) == 3 and PIXEL_SHAPE[2] != 3):
    raise ValueError("PIXEL_SHAPE harus (tinggi, lebar) atau (tinggi, lebar, 3).")
if len(pixel_cols) != int(np.prod(PIXEL_SHAPE)) or PIXEL_MAX <= 0:
    raise ValueError("Periksa jumlah kolom, PIXEL_SHAPE, dan PIXEL_MAX.")
pixels = df_table[pixel_cols].apply(pd.to_numeric, errors="coerce").to_numpy()
valid = np.isfinite(pixels).all(axis=1) & (pixels >= 0).all(axis=1) & (pixels <= PIXEL_MAX).all(axis=1)
arrays = [np.rint(row.reshape(PIXEL_SHAPE) / PIXEL_MAX * 255).astype(np.uint8) for row in pixels[valid]]
row_ids = df_table.index[valid]
labels = df_table.loc[valid, LABEL_COLUMN].to_numpy() if LABEL_COLUMN is not None else [None] * len(arrays)
df_inputs = pd.DataFrame({"image": arrays, "path": [f"row_{i}" for i in row_ids], "label": labels})
load_failed = [{"path": f"row_{i}", "error": "Piksel hilang/non-numerik/di luar rentang"} for i in df_table.index[~valid]]
print("Baris valid:", len(df_inputs), "| Baris gagal:", len(load_failed))


## CHUNK E — Folder gambar dari Kaggle

Ganti `KAGGLE_DATASET` dengan slug dari URL `kaggle.com/datasets/pemilik/nama-dataset`. Chunk ini untuk dataset berisi file gambar; jika dataset berisi CSV piksel, pilih chunk G.


In [ ]:
# %pip install kagglehub
import kagglehub

KAGGLE_DATASET = "pemilik/nama-dataset"  # wajib diganti sesuai dataset soal
IMAGE_SUBFOLDER = ""  # contoh: "train" atau "dataset/train"
LABEL_FROM_FOLDER = False

dataset_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print("Lokasi dataset:", dataset_path)
print("Isi dataset:", [p.name for p in dataset_path.iterdir()][:20])
DATA_DIR = dataset_path / IMAGE_SUBFOLDER


In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Folder tidak ditemukan: {DATA_DIR}")
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
paths = sorted(p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
labels = []
for file in paths:
    parts = file.relative_to(DATA_DIR).parts
    labels.append(parts[0] if LABEL_FROM_FOLDER and len(parts) > 1 else None)
df_inputs = pd.DataFrame({"image": paths, "path": [str(p) for p in paths], "label": labels})
load_failed = []
print("Gambar ditemukan:", len(df_inputs))
display(df_inputs[["path", "label"]].head())


## CHUNK F — MNIST dari TensorFlow

MNIST merupakan alternatif tersendiri. Hanya train dipakai untuk clustering; test disimpan terpisah. Koneksi diperlukan saat download pertama.


In [ ]:
# %pip install tensorflow
import tensorflow as tf

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
selected = np.arange(len(X_train))
if MAX_ITEMS is not None and len(selected) > MAX_ITEMS:
    selected = np.random.default_rng(RANDOM_STATE).choice(selected, MAX_ITEMS, replace=False)
df_inputs = pd.DataFrame({"image": [X_train[i] for i in selected],
                          "path": [f"mnist_train_{i}" for i in selected],
                          "label": y_train[selected]})
load_failed = []
print("Train dipakai:", len(df_inputs), "| Test terpisah:", len(X_test))


## CHUNK G — CSV/Excel piksel dari Kaggle

Download dan pembacaan piksel berada dalam satu chunk lengkap. Ganti slug, nama file, bentuk gambar, serta kolom label/piksel. Hapus semua alternatif lain, termasuk D dan E.


In [ ]:
# %pip install kagglehub
import kagglehub

KAGGLE_DATASET = "pemilik/nama-dataset"  # wajib diganti sesuai dataset soal
FILE_IN_DATASET = "data.csv"
dataset_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print("Lokasi dataset:", dataset_path)
print("Contoh isi:", [str(p.relative_to(dataset_path)) for p in dataset_path.rglob("*") if p.is_file()][:20])


In [ ]:
TABLE_PATH = dataset_path / FILE_IN_DATASET
LABEL_COLUMN = None
CSV_SEP = ","
CSV_ENCODING = "utf-8-sig"
PIXEL_SHAPE = (28, 28)  # (tinggi, lebar) atau (tinggi, lebar, 3)
PIXEL_COLUMNS = None  # daftar nama kolom piksel; None memilih selain kolom yang dikecualikan
NON_PIXEL_COLUMNS = ["id", "Id"]
PIXEL_MAX = 255.0

if TABLE_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df_table = pd.read_excel(TABLE_PATH)
else:
    df_table = pd.read_csv(TABLE_PATH, sep=CSV_SEP, encoding=CSV_ENCODING)
display(df_table.head())
if LABEL_COLUMN is not None and LABEL_COLUMN not in df_table.columns:
    raise ValueError("LABEL_COLUMN tidak ditemukan.")
if MAX_ITEMS is not None and len(df_table) > MAX_ITEMS:
    print("Sampling tabel piksel:", MAX_ITEMS, "dari", len(df_table))
    df_table = df_table.sample(n=MAX_ITEMS, random_state=RANDOM_STATE)


In [ ]:
excluded = NON_PIXEL_COLUMNS + ([LABEL_COLUMN] if LABEL_COLUMN is not None else [])
pixel_cols = PIXEL_COLUMNS if PIXEL_COLUMNS is not None else [c for c in df_table.columns if c not in excluded]
if len(PIXEL_SHAPE) not in {2, 3} or (len(PIXEL_SHAPE) == 3 and PIXEL_SHAPE[2] != 3):
    raise ValueError("PIXEL_SHAPE harus (tinggi, lebar) atau (tinggi, lebar, 3).")
if len(pixel_cols) != int(np.prod(PIXEL_SHAPE)) or PIXEL_MAX <= 0:
    raise ValueError("Periksa jumlah kolom, PIXEL_SHAPE, dan PIXEL_MAX.")
pixels = df_table[pixel_cols].apply(pd.to_numeric, errors="coerce").to_numpy()
valid = np.isfinite(pixels).all(axis=1) & (pixels >= 0).all(axis=1) & (pixels <= PIXEL_MAX).all(axis=1)
arrays = [np.rint(row.reshape(PIXEL_SHAPE) / PIXEL_MAX * 255).astype(np.uint8) for row in pixels[valid]]
row_ids = df_table.index[valid]
labels = df_table.loc[valid, LABEL_COLUMN].to_numpy() if LABEL_COLUMN is not None else [None] * len(arrays)
df_inputs = pd.DataFrame({"image": arrays, "path": [f"row_{i}" for i in row_ids], "label": labels})
load_failed = [{"path": f"row_{i}", "error": "Piksel hilang/non-numerik/di luar rentang"} for i in df_table.index[~valid]]
print("Baris valid:", len(df_inputs), "| Baris gagal:", len(load_failed))


# PREPROCESSING BERSAMA — semua input lanjut dari sini

Pertahankan seluruh cell mulai bagian ini. Semua chunk menyediakan `df_inputs` dengan
kolom `image`, `path`, `label`. Kolom `image` berisi lokasi file atau array piksel, sehingga
proses resize/channel/normalisasi dapat memakai alur yang sama.


In [ ]:
if COLOR_MODE not in {"L", "RGB"}:
    raise ValueError("COLOR_MODE harus L atau RGB.")
if df_inputs.empty:
    raise ValueError("Tidak ada gambar. Periksa lokasi file/folder dan ekstensi.")
total_inputs = len(df_inputs)
if MAX_ITEMS is not None and len(df_inputs) > MAX_ITEMS:
    df_inputs = df_inputs.sample(n=MAX_ITEMS, random_state=RANDOM_STATE)
df_inputs = df_inputs.reset_index(drop=True)
print("Gambar diproses:", len(df_inputs), "dari", total_inputs)


## Baca gambar, samakan ukuran dan channel

Orientasi EXIF diperbaiki. Transparansi memakai latar putih; file rusak dicatat.
Template ini memakai gambar tampilan 8-bit. Data medis/16-bit atau multipage perlu
preprocessing sesuai formatnya. Metadata dan gambar selalu ditambahkan bersama.


In [ ]:
images, records = [], []
failed = load_failed.copy()

for row in df_inputs.itertuples(index=False):
    try:
        if isinstance(row.image, (str, Path)):
            with Image.open(row.image) as file_image:
                img = ImageOps.exif_transpose(file_image).copy()
        else:
            img = Image.fromarray(row.image)
        original_size = img.size
        if "A" in img.getbands() or "transparency" in img.info:
            rgba = img.convert("RGBA")
            img = Image.alpha_composite(Image.new("RGBA", rgba.size, "white"), rgba)
        img = img.convert(COLOR_MODE).resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
        images.append(np.array(img))
        records.append({"path": row.path, "label": row.label,
                        "original_width": original_size[0], "original_height": original_size[1]})
    except (OSError, ValueError, TypeError) as error:
        failed.append({"path": row.path, "error": str(error)})


## 5. Cek data dan normalisasi
Satu gambar menjadi satu baris metadata. Piksel dinormalisasi ke 0–1; pembagian dilakukan tepat sekali.


In [ ]:
df_failed = pd.DataFrame(failed, columns=["path", "error"])
print("Gagal diproses:", len(df_failed))
if not df_failed.empty:
    display(df_failed.head(10))
if not images:
    raise ValueError("Tidak ada gambar valid yang berhasil dibaca.")

X_images = np.stack(images).astype(np.float32) / 255.0
df_data = pd.DataFrame(records).reset_index(drop=True)
print("Shape gambar:", X_images.shape)
print("Rentang piksel:", X_images.min(), "sampai", X_images.max())
display(df_data.head())


In [ ]:
num = min(12, len(X_images))
fig, axes = plt.subplots(int(np.ceil(num / 4)), 4, figsize=(9, 2.5 * int(np.ceil(num / 4))))
for i, ax in enumerate(np.atleast_1d(axes).ravel()):
    if i < num:
        ax.imshow(X_images[i], cmap="gray" if COLOR_MODE == "L" else None, vmin=0, vmax=1)
        ax.set_title(f"Data {i} | {df_data.loc[i, 'label']}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 6. Flatten gambar

`(n, tinggi, lebar)` menjadi `(n, tinggi × lebar)`; RGB menambahkan faktor 3.
Ini baseline berbasis piksel seperti notebook contoh. Jarak piksel sensitif terhadap posisi,
pencahayaan, dan latar; cluster dapat mencerminkan faktor tersebut, bukan objek semantik.
StandardScaler per piksel tidak dipakai secara otomatis karena dapat memperbesar pengaruh
piksel yang variansnya sangat kecil.


In [ ]:
X_flattened = X_images.reshape(len(X_images), -1)
print("Shape fitur:", X_flattened.shape)


## Clustering — pilih k atau ikuti k dari soal

Ubah `K_FIXED` di konfigurasi jika soal menentukan jumlah cluster. Jika `None`,
bandingkan `K_CANDIDATES` dan pilih silhouette tertinggi pada kandidat yang dicoba.
Ini pemilihan eksploratif pada data yang sama, bukan skor generalisasi test.

`X_model` adalah fitur yang dipakai K-Means. Silhouette dihitung pada ruang fitur ini,
bukan pada koordinat grafik dua dimensi. Untuk data besar, skor menggunakan maksimal
`SILHOUETTE_SAMPLE` baris acak yang sama pada setiap k. Skor sampel merupakan perkiraan.


In [ ]:
X_model = X_flattened

n_samples = X_model.shape[0]
if n_samples < 3:
    raise ValueError("Perlu minimal 3 data valid untuk alur clustering dan silhouette ini.")

k_values = [K_FIXED] if K_FIXED is not None else K_CANDIDATES
k_values = sorted(set(k for k in k_values if 2 <= k < n_samples))
if not k_values:
    raise ValueError("K harus minimal 2 dan lebih kecil dari jumlah data. Periksa konfigurasi.")

rng = np.random.default_rng(RANDOM_STATE)
eval_size = n_samples if SILHOUETTE_SAMPLE is None else min(SILHOUETTE_SAMPLE, n_samples)
if eval_size < 3:
    raise ValueError("SILHOUETTE_SAMPLE harus minimal 3, atau None.")
eval_index = rng.choice(n_samples, size=eval_size, replace=False)
X_eval = X_model[eval_index]
print("Data modeling:", X_model.shape, "| Data evaluasi silhouette:", eval_size)


In [ ]:
results = []
models = {}

for k in k_values:
    if USE_MINIBATCH:
        model = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE,
                                n_init=10, batch_size=1024)
    else:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)

    labels = model.fit_predict(X_model)
    eval_labels = labels[eval_index]
    score = np.nan

    if 2 <= len(np.unique(eval_labels)) < len(eval_labels):
        score = silhouette_score(X_eval, eval_labels, metric="euclidean")

    models[k] = model
    results.append({"k": k, "cluster_terbentuk": len(np.unique(labels)),
                    "silhouette": score, "inertia": model.inertia_})
    print(f"k={k} | silhouette={score:.4f} | inertia={model.inertia_:.2f}")

df_scores = pd.DataFrame(results)
display(df_scores)


`NaN` pada silhouette berarti sampel evaluasi tidak memenuhi syarat jumlah cluster.
Jika cluster sangat kecil terlewat oleh sampling, naikkan `SILHOUETTE_SAMPLE` atau
gunakan `None`. Data identik dapat menghasilkan lebih sedikit cluster daripada k.


In [ ]:
if K_FIXED is not None:
    best_k = K_FIXED
else:
    valid_scores = df_scores.dropna(subset=["silhouette"])
    if valid_scores.empty:
        raise ValueError("Tidak ada silhouette valid. Periksa data identik atau ukuran sampel evaluasi.")
    best_k = int(valid_scores.loc[valid_scores["silhouette"].idxmax(), "k"])

kmeans = models[best_k]
df_result = df_data.copy()
df_result["cluster"] = kmeans.predict(X_model)
print("K yang dipakai:", best_k)
display(df_result.head())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(df_scores["k"], df_scores["silhouette"], marker="o")
axes[0].set_title("Silhouette per k")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Silhouette")

axes[1].plot(df_scores["k"], df_scores["inertia"], marker="o")
axes[1].set_title("Elbow / inertia")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Inertia")
plt.tight_layout()
plt.show()


## Distribusi dan evaluasi tambahan

Silhouette mendekati 1 berarti pemisahan lebih baik; sekitar 0 berarti cluster
bertumpang tindih; nilai negatif menunjukkan sebagian data lebih dekat ke cluster lain.
Inertia biasanya turun ketika k naik; cari perubahan kemiringan untuk membantu membaca elbow.

ARI dan NMI hanya dihitung jika ada label acuan yang bermakna. Nomor cluster tidak
otomatis sama dengan label kelas. Jangan langsung memakai accuracy pada nomor cluster.


In [ ]:
cluster_count = df_result["cluster"].value_counts().sort_index()
display(cluster_count.rename("jumlah").to_frame())

cluster_count.plot(kind="bar", figsize=(7, 4))
plt.xlabel("Cluster")
plt.ylabel("Jumlah data")
plt.title("Distribusi cluster")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
label_mask = df_result["label"].notna()
if label_mask.sum() >= 2 and df_result.loc[label_mask, "label"].nunique() >= 2:
    y_true = df_result.loc[label_mask, "label"].astype(str)
    y_cluster = df_result.loc[label_mask, "cluster"]

    print("Jumlah data dengan label acuan:", len(y_true))
    print("ARI:", round(adjusted_rand_score(y_true, y_cluster), 4))
    print("NMI:", round(normalized_mutual_info_score(y_true, y_cluster), 4))
    display(pd.crosstab(y_true, y_cluster, rownames=["Label acuan"], colnames=["Cluster"]))
else:
    print("ARI/NMI dilewati: label acuan tidak tersedia atau hanya ada satu kelas.")


## Visualisasi PCA

PCA di sini hanya untuk visualisasi. K-Means dan silhouette tetap memakai `X_model`.
Explained variance menunjukkan berapa proporsi variasi fitur yang ditangkap grafik.
Titik yang berdekatan pada grafik belum tentu berdekatan pada semua dimensi asal.


In [ ]:
pca = PCA(n_components=min(2, X_model.shape[0], X_model.shape[1]), random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_model)
plot_y = X_pca[:, 1] if X_pca.shape[1] == 2 else np.zeros(len(X_pca))

plt.figure(figsize=(8, 6))
for cluster in sorted(df_result["cluster"].unique()):
    mask = df_result["cluster"].to_numpy() == cluster
    plt.scatter(X_pca[mask, 0], plot_y[mask], label=f"Cluster {cluster}", alpha=0.6, s=20)
plt.xlabel("PC1")
plt.ylabel("PC2" if X_pca.shape[1] == 2 else "Tidak ada komponen kedua")
plt.title("Visualisasi cluster dengan PCA")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print("Total explained variance:", round(pca.explained_variance_ratio_.sum(), 4))


## Interpretasi — centroid
Centroid adalah rata-rata piksel anggota cluster. Bentuk yang kabur menunjukkan variasi posisi/bentuk anggota.


In [ ]:
centers = kmeans.cluster_centers_.reshape((best_k,) + X_images.shape[1:])
n_cols = min(5, best_k)
n_rows = int(np.ceil(best_k / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 2.5 * n_rows))
for cluster, ax in enumerate(np.atleast_1d(axes).ravel()):
    if cluster < best_k:
        ax.imshow(np.clip(centers[cluster], 0, 1), cmap="gray" if COLOR_MODE == "L" else None, vmin=0, vmax=1)
        ax.set_title(f"Centroid {cluster}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## Interpretasi — contoh anggota cluster
Tampilkan beberapa anggota untuk memeriksa isi cluster, lalu jelaskan pola visual yang terlihat.


In [ ]:
for cluster in sorted(df_result["cluster"].unique()):
    indices = np.flatnonzero(df_result["cluster"].to_numpy() == cluster)[:5]
    fig, axes = plt.subplots(1, len(indices), figsize=(2.5 * len(indices), 2.5))
    for ax, index in zip(np.atleast_1d(axes), indices):
        ax.imshow(X_images[index], cmap="gray" if COLOR_MODE == "L" else None, vmin=0, vmax=1)
        ax.set_title(f"Data {index}")
        ax.axis("off")
    fig.suptitle(f"Cluster {cluster}")
    plt.tight_layout()
    plt.show()


## Simpan hasil (opsional)

Ubah `SAVE_RESULTS = True` jika ingin membuat CSV. Semua baris hasil mengikuti urutan
data yang benar-benar berhasil diproses. File gagal tersedia terpisah pada `df_failed`
jika notebook ini membaca kumpulan file.


In [ ]:
SAVE_RESULTS = False
OUTPUT_DIR = Path("hasil_uts")

if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df_result.to_csv(OUTPUT_DIR / "image_hasil.csv", index=False)
    df_scores.to_csv(OUTPUT_DIR / "image_skor_k.csv", index=False)
    print("Hasil disimpan di:", OUTPUT_DIR.resolve())


## Catatan saat UTS

- Folder kategori: pastikan label bukan nama folder pembungkus seperti `train`/`test`.
- CSV lokasi harus merujuk file yang tersedia di komputer, bukan path komputer pembuat dataset.
- Pada soal train/test, fit K-Means pada train dan gunakan `predict` pada test; gunakan
  resize, channel, dan normalisasi yang sama. PCA untuk test juga memakai `transform`, bukan fit ulang.
- Untuk MNIST dengan `IMAGE_SIZE=(28,28)` dan `COLOR_MODE="L"`, contoh prediksi test:
  `test_clusters = kmeans.predict(X_test.reshape(len(X_test), -1).astype(np.float32) / 255.0)`.
  Jika ukuran/channel diubah, proses ulang gambar test dengan pengaturan yang sama terlebih dahulu.

- Jika soal menetapkan k, ubah `K_FIXED`, lalu jalankan kembali bagian clustering dan hasil setelahnya.
- Jika chunk loading atau preprocessing diubah, jalankan ulang chunk terpilih sampai akhir.
- `MAX_ITEMS = None` berarti memakai seluruh data; angka tertentu berarti sampel acak yang dilaporkan.
- Untuk data besar, coba `USE_MINIBATCH = True` bila soal mengizinkan MiniBatch K-Means;
  hasilnya dapat berbeda dari K-Means biasa. Silhouette sampling hanya mempercepat evaluasi,
  sedangkan `MAX_ITEMS` atau MiniBatch membantu mengurangi beban modeling.
- Sesuaikan metode dengan instruksi soal. Template ini berfokus pada clustering seperti notebook contoh.
- Hasil k terbaik hanya terbaik di antara kandidat dan representasi yang dicoba; tidak menjamin cluster
  tersebut paling bermakna. Jelaskan juga isi cluster dan keterbatasan fitur.


Referensi API: [Pillow Image](https://pillow.readthedocs.io/en/stable/reference/Image.html),
[KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html),
[silhouette_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html).

Kaggle: [dokumentasi resmi kagglehub](https://github.com/Kaggle/kagglehub).
